<a href="https://colab.research.google.com/github/annikaks/CS329XHumanCenteredLLMs-Project/blob/main/sft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import json
import random
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)

# Load the anonymized datasets
with open("/content/friends_AB_anonymized.json", encoding="utf-8") as f:
    scenes_AB = json.load(f)

with open("/content/friends_AC_anonymized.json", encoding="utf-8") as f:
    scenes_AC = json.load(f)

print("A-B segments:", len(scenes_AB))
print("A-C segments:", len(scenes_AC))

# Collect unique episode/conversation IDs across BOTH datasets
all_scenes = scenes_AB + scenes_AC
episode_ids = sorted({
    scene["conversation_id"] for scene in all_scenes
})

train_ids, test_ids = train_test_split(
    episode_ids,
    test_size=0.20,
    random_state=SEED
)

train_ids = set(train_ids)
test_ids = set(test_ids)

# Split both datasets using the same episode-level split
train_AB = [s for s in scenes_AB if s["conversation_id"] in train_ids]
test_AB  = [s for s in scenes_AB if s["conversation_id"] in test_ids]

train_AC = [s for s in scenes_AC if s["conversation_id"] in train_ids]
test_AC  = [s for s in scenes_AC if s["conversation_id"] in test_ids]

print("\nTraining A-B segments:", len(train_AB))
print("Testing A-B segments:", len(test_AB))
print("Training A-C segments:", len(train_AC))
print("Testing A-C segments:", len(test_AC))

# Verify that no episode appears in both splits
assert train_ids.isdisjoint(test_ids)

# Save the split so it can be reused consistently
with open("/content/friends_split.json", "w", encoding="utf-8") as f:
    json.dump({
        "train_episode_ids": sorted(train_ids),
        "test_episode_ids": sorted(test_ids)
    }, f, indent=2)

print("\nEpisode-level split saved.")

A-B segments: 481
A-C segments: 295

Training A-B segments: 383
Testing A-B segments: 98
Training A-C segments: 240
Testing A-C segments: 55

Episode-level split saved.


In [3]:
!pip -q install -U transformers datasets accelerate peft trl bitsandbytes "pyarrow<24,>=19.0.0"

In [12]:
import re
from datasets import Dataset

MAX_CONTEXT_TURNS = 6

def make_examples(scenes, max_context_turns=MAX_CONTEXT_TURNS):
    examples = []
    for scene in scenes:
        pair = scene["pair"]
        partner = next((p for p in pair if p != "Person A"), None)
        if partner not in {"Person B", "Person C"}:
            continue

        turns = scene["turns"]
        for i, turn in enumerate(turns):
            if turn["speaker"] != "Person A" or i == 0:
                continue

            gold_text = turn.get("text", "").strip()
            if not gold_text:
                continue  # Skip empty Person A turns

            context = turns[max(0, i - max_context_turns):i]
            examples.append({
                "conversation_id": scene["conversation_id"],
                "partner": partner,
                "context": [
                    {"speaker": t["speaker"], "text": t["text"]}
                    for t in context
                ],
                "gold_response": gold_text
            })
    return examples

def format_turns_masked(turns):
    lines = []
    for turn in turns:
        speaker = "[Other Person]" if turn["speaker"] != "Person A" else "Person A"
        text = re.sub(
            r"\bPerson [BC]\b",
            "[Other Person]",
            turn["text"],
            flags=re.IGNORECASE
        )
        lines.append(f"{speaker}: {text}")
    return "\n".join(lines)

def build_instruction(context_turns):
    context_str = format_turns_masked(context_turns)
    return f"""You are generating dialogue as Person A.

Respond naturally as Person A based on the conversation history.
Generate exactly one response from Person A without any speaker label.

CONVERSATION:
{context_str}

Person A:"""

def build_messages(context_turns, assistant_reply=None):
    messages = [{"role": "user", "content": build_instruction(context_turns)}]
    if assistant_reply is not None:
        messages.append({"role": "assistant", "content": assistant_reply})
    return messages

def format_sft_example(example):
    return {
        "messages": build_messages(example["context"], example["gold_response"])
    }

# Extract turn-level examples identically to BaselineRun.ipynb
raw_train_examples = make_examples(train_AB) + make_examples(train_AC)
test_examples = make_examples(test_AB) + make_examples(test_AC)

all_train_formatted = [format_sft_example(ex) for ex in raw_train_examples]

random.seed(SEED)
random.shuffle(all_train_formatted)

# Hold out 10% of training examples for validation loss
val_size = int(0.1 * len(all_train_formatted))
val_dataset = Dataset.from_list(all_train_formatted[:val_size])
train_dataset = Dataset.from_list(all_train_formatted[val_size:])

print(f"SFT Train examples: {len(train_dataset)} | SFT Val examples: {len(val_dataset)}")
print(f"SFT Test examples:  {len(test_examples)}")
print("\nSample messages:\n", train_dataset[0]["messages"])

SFT Train examples: 2006 | SFT Val examples: 222
SFT Test examples:  535

Sample messages:
 [{'content': "You are generating dialogue as Person A.\n\nRespond naturally as Person A based on the conversation history.\nGenerate exactly one response from Person A without any speaker label.\n\nCONVERSATION:\n[Other Person]: OK, see ya later, nice meeting you. You're welcome.\n\nPerson A:", 'role': 'user'}, {'content': 'What?', 'role': 'assistant'}]


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [13]:
import torch
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

SAVE_DIR = "/content/drive/MyDrive/329X_Project/Results/mistral_friends_sft_adapter_v2"

model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

sft_config = SFTConfig(
    output_dir="/content/sft_checkpoints_v2",
    max_length=1024,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=6,
    fp16=False,
    bf16=False,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=sft_config,
)

trainer.train()

trainer.model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved SFT adapter weights to: {SAVE_DIR}")

def build_sft_prompt(context_turns):
    return tokenizer.apply_chat_template(
        build_messages(context_turns),
        tokenize=False,
        add_generation_prompt=True,
    )

# Immediate 5-example sanity check while model is in memory
trainer.model.eval()
trainer.model.config.use_cache = True
for ex in test_examples[:5]:
    prompt = build_sft_prompt(ex["context"])
    inputs = tokenizer(prompt, return_tensors="pt").to(trainer.model.device)
    with torch.no_grad():
        out = trainer.model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    reply = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    print("GOLD:", ex["gold_response"])
    print("GEN: ", reply)
    print("-" * 40)

/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:148: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:342: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Tokenizing train dataset:   0%|          | 0/2006 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/2006 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2006 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/2006 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/222 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/222 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/222 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/222 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
25,1.344467,1.211374,1.205068,0.718192,62776.000000
50,1.149145,1.111937,1.140910,0.737629,121022.000000
75,1.127109,1.071419,1.075212,0.745623,180998.000000
100,1.100393,1.046390,1.034555,0.750338,242849.000000
125,1.069760,1.042297,1.041321,0.751409,304013.000000
126,1.069760,1.042292,1.041323,0.751360,305223.000000


Saved SFT adapter weights to: /content/drive/MyDrive/329X_Project/Results/mistral_friends_sft_adapter_v2
GOLD: Oh, big glamour night. Me and Person C at Laundorama.
GEN:  Oh, I don't know. What do you have in mind?
----------------------------------------
GOLD: Who?
GEN:  Who?
----------------------------------------
GOLD: Don't you have a laundry room in your building?
GEN:  Sure!
----------------------------------------
GOLD: Sure.
GEN:  No, no-no-no, we can't do that. We'll never see each other again.
----------------------------------------
GOLD: Hi, uh, nothing. That horrible woman just took my machine.
GEN:  Well, I had a really weird dream about you last night...
----------------------------------------


In [14]:
import os
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
ADAPTER_DIR = "/content/drive/MyDrive/329X_Project/Results/mistral_friends_sft_adapter_v2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
)

sft_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
sft_model.eval()
sft_model.config.use_cache = True

def generate_sft_response(context_turns, max_new_tokens=100):
    formatted_prompt = build_sft_prompt(context_turns)
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(sft_model.device)

    with torch.no_grad():
        output = sft_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = output[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    if response.lower().startswith("person a:"):
        response = response[len("person a:"):].strip()

    return formatted_prompt, response

sft_results = []
for i, example in enumerate(test_examples):
    prompt, generated = generate_sft_response(example["context"])
    sft_results.append({
        "conversation_id": example["conversation_id"],
        "actual_partner": example["partner"],
        "prompt": prompt,
        "context": example["context"],
        "gold_response": example["gold_response"],
        "generated_response": generated
    })
    if (i + 1) % 25 == 0 or i + 1 == len(test_examples):
        print(f"Generated {i + 1}/{len(test_examples)} responses")

results_dir = "/content/drive/MyDrive/329X_Project/Results"
os.makedirs(results_dir, exist_ok=True)

json_path = os.path.join(results_dir, "mistral_sft_test_results.json")
txt_path = os.path.join(results_dir, "mistral_sft_prompts_and_responses.txt")

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(sft_results, f, ensure_ascii=False, indent=2)

with open(txt_path, "w", encoding="utf-8") as f:
    for i, res in enumerate(sft_results, start=1):
        f.write("=" * 80 + "\n")
        f.write(f"EXAMPLE {i} | ID: {res['conversation_id']} | Actual partner: {res['actual_partner']}\n")
        f.write("=" * 80 + "\n\n")
        f.write("PROMPT:\n" + res["prompt"] + "\n\n")
        f.write("GOLD RESPONSE:\n" + res["gold_response"] + "\n\n")
        f.write("GENERATED RESPONSE:\n" + res["generated_response"] + "\n\n\n")

print(f"Saved JSON to: {json_path}")
print(f"Saved TXT to:  {txt_path}")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Generated 25/535 responses
Generated 50/535 responses
Generated 75/535 responses
Generated 100/535 responses
Generated 125/535 responses
Generated 150/535 responses
Generated 175/535 responses
Generated 200/535 responses
Generated 225/535 responses
Generated 250/535 responses
Generated 275/535 responses
Generated 300/535 responses
Generated 325/535 responses
Generated 350/535 responses
Generated 375/535 responses
Generated 400/535 responses
Generated 425/535 responses
Generated 450/535 responses
Generated 475/535 responses
Generated 500/535 responses
Generated 525/535 responses
Generated 535/535 responses
Saved JSON to: /content/drive/MyDrive/329X_Project/Results/mistral_sft_test_results.json
Saved TXT to:  /content/drive/MyDrive/329X_Project/Results/mistral_sft_prompts_and_responses.txt


In [ ]:
from safetensors.torch import load_file
import torch

weights = load_file("/content/drive/MyDrive/329X_Project/Results/mistral_friends_sft_adapter/adapter_model.safetensors")
for k, v in list(weights.items())[:5]:
    print(k, v.dtype, v.abs().mean().item(), torch.isnan(v).any().item())

base_model.model.model.layers.0.mlp.down_proj.lora_A.weight torch.float32 0.00450375908985734 False
base_model.model.model.layers.0.mlp.down_proj.lora_B.weight torch.float32 0.0015561111504212022 False
base_model.model.model.layers.0.mlp.gate_proj.lora_A.weight torch.float32 0.008032652549445629 False
base_model.model.model.layers.0.mlp.gate_proj.lora_B.weight torch.float32 0.0016292749205604196 False
base_model.model.model.layers.0.mlp.up_proj.lora_A.weight torch.float32 0.008019312284886837 False
